In [ ]:
# -*- coding: utf-8 -*-

"""
Calculate per-molecule frequency-domain susceptibility spectra
from single-molecule KWW fit parameters.

The input file should be *_final_KWWfit.csv from the ACF/KWW
analysis code.

Molecules are filtered by fit quality and sorted by tau_c from
long to short before calculating chi''(omega).

Outputs:
    Chi_per_molecule.csv                 Per-molecule chi''(omega)
    Chi_per_molecule_sorted_params.csv   Sorted KWW parameters
"""

# ============================================================
# Imports
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# User Inputs
# ============================================================

# Input KWW fit file:
#   *_final_KWWfit.csv

KWW_FILE = r"C:\path\to\final_KWWfit.csv"

# Time between frames (s)
TBF = 0.4

# Trajectory length used for the ACF/KWW analysis
N_FRAMES = 10000

R2_MIN = 0.90

N_OMEGA = 64

OUT_CSV = r"C:\path\to\Chi_per_molecule.csv"


# ============================================================
# Helper Functions
# ============================================================

def make_omega_grid(n_frames, dt, n_omega):

    total_time = n_frames * dt

    omega_min = 2 * np.pi / total_time
    omega_max = 0.95 * np.pi / dt

    return np.logspace(
        np.log10(omega_min),
        np.log10(omega_max),
        n_omega
    )


def load_good_kww_params(kww_file, r2_min):

    df = pd.read_csv(kww_file)

    good = (
        df["A"].notna()
        & df["tau_c"].notna()
        & df["tau_fit"].notna()
        & df["beta"].notna()
        & df["R2"].notna()
        & (df["A"] > 0)
        & (df["tau_c"] > 0)
        & (df["tau_fit"] > 0)
        & (df["beta"] > 0)
        & (df["beta"] <= 1.0)
        & (df["R2"] >= r2_min)
    )

    dfg = (
        df.loc[good, ["A", "tau_c", "tau_fit", "beta"]]
        .sort_values("tau_c", ascending=False)
        .reset_index(drop=True)
    )

    return dfg


def chi_loss_single(omega, A, tau_fit, beta, u):

    kernel = beta * u ** (beta - 1) * np.exp(-(u ** beta))
    wt = np.outer(omega, tau_fit * u)

    chi = A * np.trapz(
        np.sin(wt) * kernel,
        u,
        axis=1
    )

    return chi


def calculate_chi_all(omega, params_df):

    u = np.logspace(-8, 6, 6000)

    A_all = params_df["A"].to_numpy(float)
    tau_fit_all = params_df["tau_fit"].to_numpy(float)
    beta_all = params_df["beta"].to_numpy(float)

    chi_all = np.empty(
        (len(omega), len(params_df))
    )

    for j, (A, tau_fit, beta) in enumerate(
        zip(A_all, tau_fit_all, beta_all)
    ):

        chi_all[:, j] = chi_loss_single(
            omega,
            A,
            tau_fit,
            beta,
            u
        )

        if (j + 1) % 50 == 0:
            print(f"Computed {j + 1}/{len(params_df)} molecules")

    return chi_all


def save_results(omega, chi_all, params_df, out_csv):

    tau_c_all = params_df["tau_c"].to_numpy(float)

    columns = ["omega_rad_per_s"]

    columns += [
        f"chi_tauC_{tau_c_all[i]:.6g}_mol_{i:04d}"
        for i in range(len(params_df))
    ]

    out_df = pd.DataFrame(
        np.column_stack([omega, chi_all]),
        columns=columns
    )

    out_df.to_csv(out_csv, index=False)

    param_out = str(
        Path(out_csv).with_suffix("")
    ) + "_sorted_params.csv"

    params_df.to_csv(param_out, index=False)

    print(f"Saved chi data to:\n{out_csv}")
    print(f"Saved sorted parameters to:\n{param_out}")


# ============================================================
# Run Analysis
# ============================================================

def main():

    omega = make_omega_grid(
        N_FRAMES,
        TBF,
        N_OMEGA
    )

    params_df = load_good_kww_params(
        KWW_FILE,
        R2_MIN
    )

    print(
        f"Using {len(params_df)} good molecules "
        f"sorted by tau_c: long -> short"
    )

    chi_all = calculate_chi_all(
        omega,
        params_df
    )

    save_results(
        omega,
        chi_all,
        params_df,
        OUT_CSV
    )

    print("Analysis complete.")


if __name__ == "__main__":
    main()